# 🩺 Upgraded Medical Text Classifier (PubMedBERT / BioClinicalBERT)
### SmartSymptom AI — High Accuracy Medical NLP Pipeline
This notebook implements an academic-grade Clinical NLP classification pipeline:
1. **Medical Pretrained Transformer**: Uses  (pretrained on millions of biomedical papers and clinical terms).
2. **Negation & Context Preservation**: Cleans noise while keeping vital clinical phrases (e.g. "no fever", "chest tightness").
3. **Stratified Split**: Rigorous 70/15/15 Train-Validation-Test partitioning.
4. **Top-3 Differential Diagnosis**: Generates calibrated softmax probabilities.
5. **Academic Evaluation**: Generates Confusion Matrix, Macro/Micro F1-Scores, and Classification Report.


In [ ]:
# ==========================================
# 1. SETUP & DEPENDENCIES
# ==========================================
!pip install -q transformers[torch] datasets scikit-learn seaborn matplotlib huggingface_hub

import os
import torch
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import joblib
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, f1_score
from transformers import (
    AutoTokenizer, 
    AutoModelForSequenceClassification, 
    Trainer, 
    TrainingArguments,
    EarlyStoppingCallback
)

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"⚡ Running on: {device} ({torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'})")


In [ ]:
# ==========================================
# 2. DATA LOADING & CLINICAL PREPROCESSING
# ==========================================
# Download Symptom2Disease dataset if not present
if not os.path.exists('Symptom2Disease.csv'):
    !wget -q https://raw.githubusercontent.com/Ankush-Chander/Symptom2Disease/main/Symptom2Disease.csv || echo "Upload Symptom2Disease.csv or use KaggleHub"

df = pd.read_csv('Symptom2Disease.csv')
if 'Unnamed: 0' in df.columns:
    df = df.drop(columns=['Unnamed: 0'])

print(f"📊 Total samples: {len(df)}")
print(f"🩺 Number of unique diseases: {df['label'].nunique()}")
print(f"Sample distribution:")
print(df['label'].value_counts().head(5))

# Text cleaning preserving clinical negation and context
import re

def clean_medical_text(text):
    text = str(text).lower()
    # Normalize whitespaces
    text = re.sub(r'\s+', ' ', text)
    # Remove unwanted special characters but preserve punctuation relevant to sentence pauses
    text = re.sub(r'[^a-zA-Z0-9\s,\.\-]', '', text)
    return text.strip()

df['cleaned_text'] = df['text'].apply(clean_medical_text)

# Encode disease labels
le = LabelEncoder()
df['label_id'] = le.fit_transform(df['label'])
num_classes = len(le.classes_)

# Save Label Encoder
os.makedirs('saved_model', exist_ok=True)
joblib.dump(le, 'saved_model/label_encoder.pkl')
print(f"✅ {num_classes} classes encoded and saved.")


In [ ]:
# ==========================================
# 3. STRATIFIED TRAIN / VAL / TEST SPLIT
# ==========================================
# 70% Train, 15% Validation, 15% Independent Test
X_train_val, X_test, y_train_val, y_test = train_test_split(
    df['cleaned_text'].tolist(),
    df['label_id'].tolist(),
    test_size=0.15,
    random_state=42,
    stratify=df['label_id']
)

X_train, X_val, y_train, y_val = train_test_split(
    X_train_val,
    y_train_val,
    test_size=0.1765, # 0.1765 * 0.85 approx 0.15 of total
    random_state=42,
    stratify=y_train_val
)

print(f"Train size: {len(X_train)} | Val size: {len(X_val)} | Test size: {len(X_test)}")


In [ ]:
# ==========================================
# 4. TOKENIZATION & MODEL INITIALIZATION
# ==========================================
# We use PubMedBERT (trained on PubMed biomedical abstracts) for superior clinical domain understanding
# Fallback to 'distilbert-base-uncased' if lightweight inference is required
MODEL_NAME = "microsoft/BiomedNLP-PubMedBERT-base-uncased-abstract"
# MODEL_NAME = "distilbert-base-uncased"  # Alternative lightweight option

print(f"📦 Loading tokenizer and base model: {MODEL_NAME}")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME, 
    num_labels=num_classes
).to(device)

class ClinicalDataset(torch.utils.data.Dataset):
    def __init__(self, texts, labels, tokenizer, max_len=128):
        self.encodings = tokenizer(texts, truncation=True, padding=True, max_length=max_len)
        self.labels = labels

    def __getitem__(self, idx):
        item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}
        item['labels'] = torch.tensor(self.labels[idx], dtype=torch.long)
        return item

    def __len__(self):
        return len(self.labels)

train_dataset = ClinicalDataset(X_train, y_train, tokenizer)
val_dataset = ClinicalDataset(X_val, y_val, tokenizer)
test_dataset = ClinicalDataset(X_test, y_test, tokenizer)


In [ ]:
# ==========================================
# 5. TRAINING WITH EARLY STOPPING & EVALUATION
# ==========================================
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    acc = accuracy_score(labels, preds)
    macro_f1 = f1_score(labels, preds, average='macro')
    return {'accuracy': acc, 'macro_f1': macro_f1}

training_args = TrainingArguments(
    output_dir='./checkpoints',
    eval_strategy='epoch',
    save_strategy='epoch',
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    num_train_epochs=8,
    weight_decay=0.01,
    warmup_ratio=0.1,
    load_best_model_at_end=True,
    metric_for_best_model='macro_f1',
    greater_is_better=True,
    logging_steps=20,
    report_to='none',
    fp16=torch.cuda.is_available()
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)]
)

print("🚀 Training model...")
trainer.train()

# Save final best model and tokenizer
model.save_pretrained('saved_model/best_medical_text_model')
tokenizer.save_pretrained('saved_model/best_medical_text_model')
print("✅ Best model saved to saved_model/best_medical_text_model")


In [ ]:
# ==========================================
# 6. INDEPENDENT TEST EVALUATION & CONFUSION MATRIX
# ==========================================
print("🧪 Evaluating on Independent Test Set...")
test_results = trainer.predict(test_dataset)
y_pred = np.argmax(test_results.predictions, axis=-1)
y_true = np.array(y_test)

acc = accuracy_score(y_true, y_pred)
macro_f1 = f1_score(y_true, y_pred, average='macro')

print(f"
🎯 Final Test Accuracy: {acc*100:.2f}%")
print(f"🎯 Final Test Macro F1: {macro_f1*100:.2f}%")
print("
--- Classification Report ---")
print(classification_report(y_true, y_pred, target_names=le.classes_))

# Plot Confusion Matrix
cm = confusion_matrix(y_true, y_pred)
plt.figure(figsize=(16, 14))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=le.classes_, yticklabels=le.classes_)
plt.title('Test Set Confusion Matrix — Medical Text Classifier', fontsize=16)
plt.xlabel('Predicted Disease', fontsize=12)
plt.ylabel('True Disease', fontsize=12)
plt.xticks(rotation=90)
plt.tight_layout()
plt.savefig('saved_model/confusion_matrix.png', dpi=300)
plt.show()
print("✅ Confusion matrix saved to saved_model/confusion_matrix.png")


In [ ]:
# ==========================================
# 7. INFERENCE FUNCTION WITH TOP-3 DIFFERENTIAL DIAGNOSIS
# ==========================================
def predict_symptoms(text, top_k=3):
    cleaned = clean_medical_text(text)
    inputs = tokenizer(cleaned, return_tensors='pt', truncation=True, padding=True, max_length=128).to(device)
    model.eval()
    with torch.no_grad():
        logits = model(**inputs).logits
        probs = torch.softmax(logits, dim=-1).cpu().numpy()[0]
    
    top_indices = np.argsort(probs)[::-1][:top_k]
    results = [
        {
            'disease': le.inverse_transform([idx])[0],
            'confidence': float(probs[idx]),
            'percentage': f"{probs[idx]*100:.1f}%"
        }
        for idx in top_indices
    ]
    return {
        'input': text,
        'primary_diagnosis': results[0]['disease'],
        'confidence': results[0]['confidence'],
        'top_candidates': results
    }

# Test Examples
examples = [
    "I have a dry cough, shortness of breath, high fever, and sudden loss of smell and taste.",
    "Severe itching on the skin, red ring-shaped rash with raised scaly edges on the arm.",
    "Sharp stabbing chest pain radiating to my left arm, cold sweat, and nausea."
]

for ex in examples:
    pred = predict_symptoms(ex)
    print(f"
Input: '{pred['input']}'")
    print(f"Top Diagnosis: {pred['primary_diagnosis']} ({pred['top_candidates'][0]['percentage']})")
    print(f"Top 3: {[c['disease'] + ' (' + c['percentage'] + ')' for c in pred['top_candidates']]}")
